# 08 Delhi buses: what exists and what is blocked

Question (T165 only): what Delhi bus data can the product show today? **None.** The only source,
S38 Delhi Open Transit Data, is blocked on owner step H04 (registration, key and Terms), so no
static GTFS, no position sample and no field list were read: there are no scheduled, live or
predicted markers and no ETA. M12 (arrival and delay) is blocked for the same reason and shows the
word "blocked" and its planned, not started, test window; it shows no number.

So this notebook is an **audit**. It lists the S38 discovery fields with their unit status,
confirms that the disabled adapter refuses before any I/O, and writes the transit view state the
unmounted `Transit.tsx` component reads. It draws no chart because there is nothing to plot, and
it never creates a row. The rules that need data (a GPS position aged 120 s or more is never live;
ETA p10/p50/p90 only with a model release) are tested on hand-built inputs in
`tests/test_delhi_product.py` and `web/src/components/Transit.test.ts`, not shown as Delhi data.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/08_delhi_transit/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path

from atlas.analysis.prices import audit_rows, load_record, source_status
from atlas.analysis.route_exposure import blocked_card, blocked_view
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.ingest.base import SourceUnavailable
from atlas.ingest.s38 import S38

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/08_delhi_transit"))
OUT.mkdir(parents=True, exist_ok=True)
dl = load_state(Path("configs/states/dl.yaml"))
record = load_record(Path("configs/sources/s38.yaml"))
m12_config = load_record(Path("configs/models/m12.yaml"))
m12_status = json.loads(Path("models/registry/m12.json").read_text("utf-8"))


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)

In [ ]:
fields = audit_rows(record)
status = source_status(record)
by_status = Counter(r["unit_status"] for r in fields)
write_csv("fields.csv", fields)
write_csv("source_status.csv", [status])
print(status["source"], status["outcome"], "licence", status["licence"], "units", status["units"])
print(len(fields), "fields;", dict(by_status), "enabled in dl.yaml:", dl.source("S38").enabled)

In [ ]:
# The adapter refuses before any I/O; a frozen evidence record is only the argument it gets.
manifest = json.loads(Path("tests/fixtures/s38/manifest.json").read_text("utf-8"))
e = manifest["evidence"][0]
raw = RawArtifact(
    sha256=e["sha256"],
    path=Path("data/raw") / e["raw_path"],
    fetched_at=e["fetched_at"],
    media_type=e["media_type"],
    response_status=e["status"],
    source_url=e["url"],
    schema_mapping_version="s38-v1",
)
try:
    rows = S38().parse(raw, dl.source("S38"))
except SourceUnavailable as exc:
    adapter_error = str(exc)
else:
    raise AssertionError(f"S38 returned {rows.num_rows} rows; it must be blocked")
print(adapter_error)

In [ ]:
# The view the Transit component reads: unavailable, with S38's own reason and retry condition,
# and a blocked M12 card (planned window, no score). No marker of any kind exists.
view = blocked_view(record) | {"card": blocked_card(m12_status, m12_config)}
(OUT / "transit_view.json").write_text(json.dumps(view, indent=2) + "\n", "utf-8", newline="\n")
print(view["state"], "|", view["reason"], "|", view["card"]["status"], view["card"]["coverage"])

In [ ]:
summary = {
    "source": status,
    "enabled_in_dl": dl.source("S38").enabled,
    "adapter_error": adapter_error,
    "fields": len(fields),
    "fields_by_unit_status": dict(sorted(by_status.items())),
    "positions_read": 0,
    "markers": {"scheduled": 0, "live": 0, "predicted": 0},
    "view": view,
    "charts": [],
    "limits": [
        "S38 is blocked on owner step H04: no static GTFS, position sample or key was obtained.",
        "There are no scheduled, live or predicted markers; no chart is drawn.",
        "ETA p10/p50/p90 show only with a model release; M12 is blocked, so no number shows.",
        "Coverage windows come from configs/models/m12.yaml; they are planned, not started, and"
        " shift as a whole if approval is late.",
        "The 120 s rule and the ETA contract are tested on hand-built inputs only.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))